# 01 — Raw Data Exploration
Olist E-Commerce Python Analytics

Initial exploration of each raw table, table by table, before any cleaning decisions are made.
For each code block: the question being asked is in a `# Q:` comment, and what was found is in a
`# Finding:` comment placed right below the code that produced it. Full write-up lives in
`01_raw_data_audit.md`.

In [1]:
import numpy as np
import pandas as pd

## Customers

In [2]:
raw_customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')

# Q: What does the customers table look like — size, structure, nulls, duplicates?
print(raw_customers.head(5))
print(raw_customers.shape)
raw_customers.info()
print(raw_customers.isna().sum())
print(raw_customers.duplicated().sum())
raw_customers.dtypes
# Finding: 99,441 rows, 5 columns, zero nulls, zero full-row duplicates.

                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  
(99441, 5)
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count

customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

In [3]:
# Q: Is customer_id the real person, or a per-order surrogate key?
print(raw_customers['customer_unique_id'].nunique())
print(raw_customers['customer_id'].nunique())
# Finding: customer_id has 99,441 unique values (one per order) vs. customer_unique_id's
# 96,096 -- customer_id is a per-order surrogate key, not the real person.
# Always use customer_unique_id for repeat-customer / RFM analysis.

96096
99441


In [4]:
# Q: Are state and city values clean and within a sane range of categories?
print(raw_customers['customer_state'].value_counts())
print(raw_customers['customer_state'].nunique())
print(raw_customers['customer_city'].value_counts())
print(raw_customers['customer_city'].nunique())
# Finding: 27 unique state codes, all valid Brazilian states. 4,119 unique city values
# before normalization.

customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
SC     3637
BA     3380
DF     2140
ES     2033
GO     2020
PE     1652
CE     1336
PA      975
MT      907
MA      747
MS      715
PB      536
PI      495
RN      485
AL      413
SE      350
TO      280
RO      253
AM      148
AC       81
AP       68
RR       46
Name: count, dtype: int64
27
customer_city
sao paulo              15540
rio de janeiro          6882
belo horizonte          2773
brasilia                2131
curitiba                1521
                       ...  
siriji                     1
natividade da serra        1
monte bonito               1
sao rafael                 1
eugenio de castro          1
Name: count, Length: 4119, dtype: int64
4119


In [5]:
# Q: Is the zip code prefix range sane?
print(raw_customers['customer_zip_code_prefix'].min())
print(raw_customers['customer_zip_code_prefix'].max())
# Finding: range 1,001-99,990. Loaded as int64, which silently drops leading zeros on
# real 5-digit prefixes (e.g. 01001 -> 1001). Flagged for cleaning.

1003
99990


In [6]:
# Q: Does city/state text need whitespace or casing normalization?
print(raw_customers['customer_city']!=raw_customers['customer_city'].str.strip())
print(raw_customers['customer_state']!=raw_customers['customer_state'].str.strip().str.lower())

0        False
1        False
2        False
3        False
4        False
         ...  
99436    False
99437    False
99438    False
99439    False
99440    False
Name: customer_city, Length: 99441, dtype: bool
0        True
1        True
2        True
3        True
4        True
         ... 
99436    True
99437    True
99438    True
99439    True
99440    True
Name: customer_state, Length: 99441, dtype: bool


In [7]:
# Q: Any city names containing digits (possible data error)?
print(raw_customers['customer_city'][raw_customers['customer_city'].str.contains(r'\d', regex=True)])
# Finding: one city legitimately contains digits ('quilometro 14 do mutum') -- a real
# rural place name, not an error.

6333    quilometro 14 do mutum
Name: customer_city, dtype: str


## Orders

In [8]:
raw_orders=pd.read_csv('../data/raw/olist_orders_dataset.csv')

# Q: What does the orders table look like structurally?
print(raw_orders.shape)
print(raw_orders.head(5))
# Finding: 99,441 rows, 8 columns.

(99441, 8)
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00       

In [9]:
# Q: Which columns have nulls, and what does order_status look like overall?
print(raw_orders.isna().sum())
print(raw_orders.duplicated().sum())
cols=raw_orders.columns
print(cols)
print(raw_orders[cols].nunique())
print(raw_orders['order_status'].value_counts())
print(raw_orders.dtypes)
# Finding: order_approved_at (160), order_delivered_carrier_date (1,783),
# order_delivered_customer_date (2,965) nulls. order_status counts: delivered 96,478;
# shipped 1,107; canceled 625; unavailable 609; invoiced 314; processing 301; created 5;
# approved 2 -- the last two are rare categories worth bucketing carefully downstream.

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
0
Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')
order_id                         99441
customer_id                      99441
order_status                         8
order_purchase_timestamp         98875
order_approved_at                90733
order_delivered_carrier_date     81018
order_delivered_customer_date    95664
order_estimated_delivery_date      459
dtype: int64
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
process

In [10]:
# Q: Are these nulls fully explained by non-delivered statuses?
print('---')
print(raw_orders[raw_orders['order_approved_at'].isna()]['order_status'].value_counts())
print('---')
print(raw_orders[raw_orders['order_delivered_carrier_date'].isna()]['order_status'].value_counts())
print('---')
print(raw_orders[raw_orders['order_delivered_customer_date'].isna()]['order_status'].value_counts())
# Finding: yes, except 8 'delivered' orders missing a customer delivery date -- a
# genuine anomaly worth a closer look.

---
order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64
---
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64
---
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [11]:
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    raw_orders[col] = pd.to_datetime(raw_orders[col])

# Q: Which exact rows are the 'delivered but no customer-delivery-date' anomaly?
raw_orders[(raw_orders['order_delivered_customer_date'].isna()) & (raw_orders['order_status']=='delivered')]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


In [12]:
# Q: Are there any impossible / out-of-order timestamps across the order lifecycle?
print(raw_orders[raw_orders['order_purchase_timestamp']>raw_orders['order_approved_at']].shape)
print(raw_orders[raw_orders['order_approved_at']>raw_orders['order_delivered_carrier_date']].shape)
print(raw_orders[raw_orders['order_delivered_carrier_date']>raw_orders['order_delivered_customer_date']].shape)
print(raw_orders[raw_orders['order_delivered_customer_date']>raw_orders['order_estimated_delivery_date']].shape)
# Finding: order_approved_at > carrier_date: 1,359 rows. carrier_date > customer_date:
# 23 rows (physically impossible -- always an error). customer_date >
# estimated_delivery_date: 7,827 rows -- this is a real business outcome (late
# delivery), not a data error.

(0, 8)
(1359, 8)
(23, 8)
(7827, 8)


In [13]:
# Q: How severe are the approved-after-carrier violations -- normal logging noise,
# or genuine errors needing a cutoff to separate them?
gap = raw_orders['order_approved_at'] - raw_orders['order_delivered_carrier_date']
print(gap[raw_orders['order_approved_at'] > raw_orders['order_delivered_carrier_date']].describe())

gap2 = raw_orders['order_delivered_carrier_date'] - raw_orders['order_delivered_customer_date']
print(gap2[raw_orders['order_delivered_carrier_date'] > raw_orders['order_delivered_customer_date']].describe())
# Finding: gap distribution median ~17hrs, 75th pct ~1 day, max 171 days -- a flat
# cutoff is needed to isolate genuine errors from normal system-logging noise; the
# max-171-day case is extreme enough to warrant a manual look.

count                      1359
mean     1 days 00:45:07.153053
std      4 days 19:18:28.055754
min             0 days 00:00:21
25%      0 days 01:24:55.500000
50%             0 days 17:10:04
75%             1 days 01:57:24
max           171 days 05:15:22
dtype: object
count                        23
mean     3 days 06:27:18.478260
std      3 days 17:18:29.050924
min             0 days 00:23:18
25%             0 days 23:20:18
50%             1 days 15:51:52
75%      5 days 08:45:32.500000
max            16 days 02:18:29
dtype: object


## Order Items

In [14]:
raw_order_items=pd.read_csv('../data/raw/olist_order_items_dataset.csv')

# Q: What does order_items look like structurally?
print(raw_order_items.head(5))
print(raw_order_items.duplicated().sum())
print(raw_order_items.info())
print(raw_order_items.shape)
print(raw_order_items['order_id'].nunique())
print(raw_order_items['order_item_id'].nunique())
print(raw_order_items['product_id'].nunique())
raw_order_items.info()
print(raw_order_items.describe())
print(raw_order_items.isna().sum())
# Finding: 112,650 rows, 7 columns, zero nulls, zero duplicates. 98,666 unique orders,
# 32,951 unique products, up to 21 items on a single order.

                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   
3  00024acbcdf0a6daa1e931b038114c75              1   
4  00042b26cf59d7ce69dfabb4e55b4fd9              1   

                         product_id                         seller_id  \
0  4244733e06e7ecb4970a6e2683c13e61  48436dade18ac8b2bce089ec2a041202   
1  e5f2d52b802189ee658865ca93d83a8f  dd7ddc04e1b6c2c614352b383efe2d36   
2  c777355d18b72b67abbeef9df44fd0fd  5b51032eddd242adc84c38acab88f23d   
3  7634da152a4610f1595efa32f14722fc  9d7a1d34a5052409006425275ba1c2b4   
4  ac6c3623068f30de03045865e4e10089  df560393f3a51e74553ab94004ba5c87   

   shipping_limit_date   price  freight_value  
0  2017-09-19 09:45:35   58.90          13.29  
1  2017-05-03 11:05:13  239.90          19.93  
2  2018-01-18 14:48:30  199.00          17.87  
3  2018-08-15 10:10:18   12.99          12.7

In [15]:
# Q: Does converting shipping_limit_date to datetime introduce any new nulls?
raw_order_items['shipping_limit_date'] = pd.to_datetime(raw_order_items['shipping_limit_date'])
print(raw_order_items['shipping_limit_date'].isna().sum())

0


In [16]:
# Q: Any invalid (zero/negative) price values?
print(raw_order_items[raw_order_items['price']<=0])
print(raw_order_items[raw_order_items['freight_value']<0])
# Finding: price min 0.85, max 6,735 -- no zero/negative prices; freight_value min 0,
# max 409.68 -- no negative freight.

Empty DataFrame
Columns: [order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value]
Index: []
Empty DataFrame
Columns: [order_id, order_item_id, product_id, seller_id, shipping_limit_date, price, freight_value]
Index: []


In [17]:
# Q: How many items ship with freight_value == 0 (free delivery)?
print(raw_order_items[raw_order_items['freight_value']==0])
print(raw_order_items[raw_order_items['freight_value']==0].shape)
# Finding: 383 rows.

                                order_id  order_item_id  \
114     00404fa7a687c8c44ca69d42695aae73              1   
258     00a870c6c06346e85335524935c600c0              1   
483     011c899816ea29773525bd3322dbb6aa              1   
508     012b3f6ab7776a8ab3443a4ad7bef2e6              1   
509     012b3f6ab7776a8ab3443a4ad7bef2e6              2   
...                                  ...            ...   
111094  fc698f330ec7fb74859071cc6cb29772              1   
111497  fd4907109f6bac23f07064af84bec02d              1   
111649  fd95e4b85ebbb81853d4a6be3d61432b              1   
112182  fee19a0dc7358b6962a611cecf6a37b4              1   
112607  ffe73fc1d73e03fdb50e63903ddfe882              1   

                              product_id                         seller_id  \
114     53b36df67ebb7c41585e8d54d6772e08  7d13fca15225358621be4086e1eb0964   
258     aca2eb7d00ea1a7b8ebd4e68314663af  955fee9216a65b617aa5c0531780ce60   
483     53b36df67ebb7c41585e8d54d6772e08  7d13fca15225358

In [18]:
# Q: How many items have freight exceeding the item's own price?
print(raw_order_items[raw_order_items['freight_value']>raw_order_items['price']])
print(raw_order_items[raw_order_items['freight_value']>raw_order_items['price']].shape)
# Finding: 4,124 rows (~3.7% of items) -- worth checking whether this is concentrated
# in specific categories/price ranges before treating it as a settled pattern.

                                order_id  order_item_id  \
58      0025081dcf9330f9a5052ae82c6ce396              1   
80      002f98c0f7efd42638ed6100ca699b42              1   
110     003edccf16bc5ec447f592913b3df2b4              1   
125     00482f2670787292280e0a8153d82467              1   
156     00602f25bffa1dcfb71e202fbf9824fb              1   
...                                  ...            ...   
112575  ffd6f465cab015b452d398ac1f5d978a              2   
112576  ffd6f465cab015b452d398ac1f5d978a              3   
112626  fff2cdc825f9fc0ba3c04227cfa02303              1   
112638  fffb0b1a50e65c449020434fa835e078              1   
112639  fffb2ef8874127f75b52b643880fd7e0              1   

                              product_id                         seller_id  \
58      4e3f399366b0047a572b6682f9bb166e  5f3ae9136c875522250f8184f253413a   
80      d41dc2f2979f52d75d78714b378d4068  7299e27ed73d2ad986de7f7c77d919fa   
110     500870614ddcf5bd84f7d26861026c8a  ef506c96320abee

In [19]:
# Q: Is (order_id, order_item_id) a clean composite key?
print(raw_order_items.duplicated(subset=['order_id', 'order_item_id']).sum())

0


In [20]:
# Q: Is the free-freight pattern spread evenly across sellers, or concentrated?
print(raw_order_items[raw_order_items['freight_value']==0]['seller_id'].value_counts())
# Finding: concentrated in just 9 sellers, 96% from the top 4 -- looks like a
# seller-level free-shipping promotion rather than random missing data.

seller_id
7d13fca15225358621be4086e1eb0964    158
955fee9216a65b617aa5c0531780ce60     99
1f50f920176fa81dab994f9023523100     56
4869f7a5dfa277a7dca6462dcf3b52b2     56
37be5a7c751166fbc5f8ccba4119e043      9
c826c40d7b19f62a09e2d7c5e7295ee2      2
bc2ac6b95e1accce9858528ee566c17e      1
cc419e0650a3c5ba77189a1882b7556a      1
8581055ce74af1daba164fdbd55a40de      1
Name: count, dtype: int64


In [21]:
# Q: How many items per order, typically?
print(raw_order_items.groupby('order_id').size().describe())

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64


## Order Payments

In [22]:
raw_order_payments=pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
raw_order_payments.head(5)

# Q: What does the payments table look like structurally?
print(raw_order_payments.isna().sum())
print(raw_order_payments.duplicated().sum())
raw_order_payments.info()
print(raw_order_payments.describe())
# Finding: 103,886 rows, 5 columns, zero nulls, zero duplicates.

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64
0
<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 4.0 MB
       payment_sequential  payment_installments  payment_value
count       103886.000000         103886.000000  103886.000000
mean             1.092679              2.853349     154.100380
std              0.706584              2.687051     217.494064
min              1.000000              0.000000       0.000000
25%              1.

In [23]:
# Q: What does the payment_type breakdown look like?
print(raw_order_payments['payment_type'].value_counts())
# Finding: credit_card 76,795; boleto 19,784; voucher 5,775; debit_card 1,529;
# not_defined 3.

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [24]:
# Q: What do the 'not_defined' payment-type rows actually look like?
print(raw_order_payments[raw_order_payments['payment_type']=='not_defined'])

                               order_id  payment_sequential payment_type  \
51280  4637ca194b6387e2d538dc89b124b0ee                   1  not_defined   
57411  00b1cb0320190ca0daa2c88b35206009                   1  not_defined   
94427  c8c528189310eaa44a745b8d9d26908b                   1  not_defined   

       payment_installments  payment_value  
51280                     1            0.0  
57411                     1            0.0  
94427                     1            0.0  


In [25]:
# Q: Any negative payment values, and what explains the zero-value rows?
print(raw_order_payments[raw_order_payments['payment_value']<0].shape)
print(raw_order_payments[raw_order_payments['payment_value']==0].shape)
print(raw_order_payments[raw_order_payments['payment_value']==0])
# Finding: payment_value == 0 in 9 rows total -- 3 are not_defined (genuinely broken:
# unknown type AND no value), 6 are voucher (legitimate: voucher fully covered the
# order). These need separate treatment, not one blanket flag.

(0, 5)
(9, 5)
                                order_id  payment_sequential payment_type  \
19922   8bcbe01d44d147f901cd3192671144db                   4      voucher   
36822   fa65dad1b0e818e3ccc5cb0e39231352                  14      voucher   
43744   6ccb433e00daae1283ccc956189c82ae                   4      voucher   
51280   4637ca194b6387e2d538dc89b124b0ee                   1  not_defined   
57411   00b1cb0320190ca0daa2c88b35206009                   1  not_defined   
62674   45ed6e85398a87c253db47c2d9f48216                   3      voucher   
77885   fa65dad1b0e818e3ccc5cb0e39231352                  13      voucher   
94427   c8c528189310eaa44a745b8d9d26908b                   1  not_defined   
100766  b23878b3e8eb4d25a158f57d96331b18                   4      voucher   

        payment_installments  payment_value  
19922                      1            0.0  
36822                      1            0.0  
43744                      1            0.0  
51280                      1   

In [26]:
# Q: Is payment_installments always a sane, non-zero value?
print(raw_order_payments['payment_installments'].value_counts().sort_index())

payment_installments
0         2
1     52546
2     12413
3     10461
4      7098
5      5239
6      3920
7      1626
8      4268
9       644
10     5328
11       23
12      133
13       16
14       15
15       74
16        5
17        8
18       27
20       17
21        3
22        1
23        1
24       18
Name: count, dtype: int64


In [27]:
# Q: How many payment rows exist per order (can one order use multiple payment methods)?
print(raw_order_payments.groupby('order_id').size().describe())
print(raw_order_payments.groupby('order_id').size().value_counts().head())

count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64
1    96479
2     2382
3      301
4      108
5       52
Name: count, dtype: int64


In [28]:
# Q: Does the per-order payment sum reconcile with the table-wide total (sanity
# check on the groupby itself, not yet a cross-table check against order_items)?
payments_per_order = raw_order_payments.groupby('order_id')['payment_value'].sum()
print(payments_per_order.sum())
print(raw_order_payments['payment_value'].sum())

16008872.12
16008872.120000001


In [29]:
# Q: What's the full picture on payment_installments == 0?
print(raw_order_payments[raw_order_payments['payment_installments']==0])
# Finding: 2 rows, both real credit_card payments with nonzero values -- a clear
# data-entry error, correct value obviously 1.

                               order_id  payment_sequential payment_type  \
46982  744bade1fcf9ff3f31d860ace076d422                   2  credit_card   
79014  1a57108394169c0b47d8f876acc9ba2d                   2  credit_card   

       payment_installments  payment_value  
46982                     0          58.69  
79014                     0         129.94  


In [30]:
print(raw_order_payments[raw_order_payments['payment_value']==0].groupby('payment_type')['payment_type'].count())

payment_type
not_defined    3
voucher        6
Name: payment_type, dtype: int64


In [31]:
# Q: What's the real range of payment values?
print(raw_order_payments['payment_value'].min())
print(raw_order_payments['payment_value'].max())
# Note: payment total reconciles with itself trivially; NOT yet checked against
# order_items totals -- this cross-table check is the one still missing at audit
# stage (carried out later in 02/03).

0.0
13664.08


## Reviews

In [32]:
raw_order_reviews=pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
raw_order_reviews.head(5)

# Q: What does the reviews table look like structurally?
print(raw_order_reviews.shape)
print(raw_order_reviews.isna().sum())
print(raw_order_reviews.duplicated().sum())
raw_order_reviews.info()
print(raw_order_reviews.describe())
# Finding: 99,224 rows, 7 columns. review_comment_title missing in 87,656 rows,
# review_comment_message missing in 58,247 -- expected, since comments are optional.

(99224, 7)
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64
0
<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB
       review_score
count  99224.000000
mean       4.086421
std        1.347579
min        1.000000
25%        4.000000
50%        5

In [33]:
# Q: What does the review_score distribution look like?
print(raw_order_reviews['review_score'].describe)
print(raw_order_reviews['review_score'].value_counts())
# Finding: 5-star (57,328), 4-star (19,142), 1-star (11,424), 3-star (8,179),
# 2-star (3,151).

<bound method NDFrame.describe of 0        4
1        5
2        5
3        5
4        5
        ..
99219    5
99220    5
99221    5
99222    4
99223    1
Name: review_score, Length: 99224, dtype: int64>
review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64


In [34]:
# Q: Any impossible dates (answered before created) or out-of-range scores?
raw_order_reviews['review_creation_date']=pd.to_datetime(raw_order_reviews['review_creation_date'])
raw_order_reviews['review_answer_timestamp']=pd.to_datetime(raw_order_reviews['review_answer_timestamp'])

print(raw_order_reviews['review_creation_date'].isna().sum())
print(raw_order_reviews['review_answer_timestamp'].isna().sum())
print(raw_order_reviews[raw_order_reviews['review_creation_date']>raw_order_reviews['review_answer_timestamp']])
print(raw_order_reviews[raw_order_reviews['review_score']<=0])

0
0
Empty DataFrame
Columns: [review_id, order_id, review_score, review_comment_title, review_comment_message, review_creation_date, review_answer_timestamp]
Index: []
Empty DataFrame
Columns: [review_id, order_id, review_score, review_comment_title, review_comment_message, review_creation_date, review_answer_timestamp]
Index: []


In [35]:
# Q: Is the review <-> order relationship clean 1:1, or many-to-many?
print(raw_order_reviews.groupby('order_id')['review_id'].count().sort_values(ascending=False).head(10))
print(raw_order_reviews[raw_order_reviews['order_id']=='03c939fd7fd3b38f8485a0f95798f1f6'])
print(raw_order_reviews.groupby('review_id')['order_id'].count().sort_values(ascending=False).head(10))
print(raw_order_reviews[raw_order_reviews['review_id']=='4d0e6dd087008d1f992d25ef6e1f619f'])
print(raw_order_reviews['review_id'].nunique())
print(raw_order_reviews['order_id'].nunique())
# Finding: many-to-many relationship -- 551 orders have 2-3 reviews each; conversely,
# some review_ids are attached to 2-3 different order_ids with identical
# creation/answer timestamps -- worth understanding whether this is a genuine
# one-review-many-orders design (bundled checkout) or an export artifact before
# deciding how to collapse to one row per order.

order_id
03c939fd7fd3b38f8485a0f95798f1f6    3
8e17072ec97ce29f0e1f111e598b0c85    3
c88b1d1b157a9999ce368f218a407141    3
df56136b8031ecd28e200bb18e6ddb2e    3
7f13a20e25350f4a55fb2a7c9a2e8d88    2
44aa72f4eebc8c0ff7a3636f6369201b    2
7b994fc7c7528d2098eebcb19d18ce7e    2
34de5f7952fbd04a9a1e5d7001b014b5    2
6318d8c5f6857de384c0b3186d036457    2
09a38776c4e31230864f867821174daa    2
Name: review_id, dtype: int64
                              review_id                          order_id  \
8273   b04ed893318da5b863e878cd3d0511df  03c939fd7fd3b38f8485a0f95798f1f6   
51527  f4bb9d6dd4fb6dcc2298f0e7b17b8e1e  03c939fd7fd3b38f8485a0f95798f1f6   
69438  405eb2ea45e1dbe2662541ae5b47e2aa  03c939fd7fd3b38f8485a0f95798f1f6   

       review_score review_comment_title  \
8273              3                  NaN   
51527             4                  NaN   
69438             3                  NaN   

                                  review_comment_message review_creation_date  \
8273   Um pont

In [36]:
# Q: Are there whitespace-only 'non-null' comment strings masquerading as real content?
raw_order_reviews['review_comment_message'] = raw_order_reviews['review_comment_message'].replace(r'^\s*$', np.nan, regex=True)
raw_order_reviews['review_comment_title'] = raw_order_reviews['review_comment_title'].replace(r'^\s*$', np.nan, regex=True)

print((raw_order_reviews['review_comment_message'].str.strip() == '').sum())
print((raw_order_reviews['review_comment_title'].str.strip() == '').sum())

0
0


## Products

In [37]:
raw_order_products=pd.read_csv('../data/raw/olist_products_dataset.csv')
raw_order_products.head(5)

# Q: What does the products table look like structurally?
raw_order_products.isna().sum()
raw_order_products.info()
raw_order_products.describe()
print(raw_order_products['product_id'].nunique())
print(raw_order_products.shape)
print(raw_order_products.duplicated().sum())
# Finding: 32,951 rows, 9 columns.

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB
32951
(32951, 9)
0


In [38]:
print(raw_order_products['product_category_name'].nunique())
print((raw_order_products['product_category_name'].dropna().str.strip()=='').sum())

73
0


In [39]:
# Q: Are the missing-metadata rows the same rows across category/name/description/
# photos? Are dimensions separately missing too?
print(raw_order_products[raw_order_products['product_category_name'].isna()])
print(raw_order_products[raw_order_products['product_name_lenght'].isna()])
print(raw_order_products[raw_order_products['product_description_lenght'].isna()])
print(raw_order_products[raw_order_products['product_photos_qty'].isna()])
print(raw_order_products[raw_order_products['product_weight_g'].isna()])
print(raw_order_products[raw_order_products['product_length_cm'].isna()])
print(raw_order_products[raw_order_products['product_height_cm'].isna()])
print(raw_order_products[raw_order_products['product_width_cm'].isna()])
# Finding: product_category_name, product_name_lenght, product_description_lenght,
# product_photos_qty are all null together on the exact same 610 rows (correlated
# missingness) -- dimensions (weight/length/height/width) are still present on these
# rows. 2 additional rows are missing dimensions entirely.

                             product_id product_category_name  \
105    a41e356c76fab66334f36de622ecbd3a                   NaN   
128    d8dee61c2034d6d075997acef1870e9b                   NaN   
145    56139431d72cd51f19eb9f7dae4d1617                   NaN   
154    46b48281eb6d663ced748f324108c733                   NaN   
197    5fb61f482620cb672f5e586bb132eae9                   NaN   
...                                 ...                   ...   
32515  b0a0c5dd78e644373b199380612c350a                   NaN   
32589  10dbe0fbaa2c505123c17fdc34a63c56                   NaN   
32616  bd2ada37b58ae94cc838b9c0569fecd8                   NaN   
32772  fa51e914046aab32764c41356b9d4ea4                   NaN   
32852  c4ceee876c82b8328e9c293fa0e1989b                   NaN   

       product_name_lenght  product_description_lenght  product_photos_qty  \
105                    NaN                         NaN                 NaN   
128                    NaN                         NaN         

In [40]:
# Q: Any impossible zero values in the dimension/weight columns?
print(raw_order_products[raw_order_products['product_name_lenght']==0])
print(raw_order_products[raw_order_products['product_description_lenght']==0])
print(raw_order_products[raw_order_products['product_photos_qty']==0])
print(raw_order_products[raw_order_products['product_weight_g']==0])
print(raw_order_products[raw_order_products['product_length_cm']==0])
print(raw_order_products[raw_order_products['product_height_cm']==0])
print(raw_order_products[raw_order_products['product_width_cm']==0])
# Finding: product_weight_g == 0: 4 rows, all in cama_mesa_banho, with real dimensions
# present -- physically impossible, flagged for imputation.

Empty DataFrame
Columns: [product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm]
Index: []
Empty DataFrame
Columns: [product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm]
Index: []
Empty DataFrame
Columns: [product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm]
Index: []
                             product_id product_category_name  \
9769   81781c0fed9fe1ad6e8c81fca1e1cb08       cama_mesa_banho   
13683  8038040ee2a71048d4bdbbdc985b69ab       cama_mesa_banho   
14997  36ba42dd187055e1fbe943b2d11430ca       cama_mesa_banho   
32079  e673e90efa65a5409ff4196c038bb5af       cama_mesa_banho   

       product_name_lenght  prod

In [41]:
# Q: Any negative values (should be none)?
print(raw_order_products[raw_order_products['product_name_lenght']<0])
print(raw_order_products[raw_order_products['product_description_lenght']<0])
print(raw_order_products[raw_order_products['product_photos_qty']<0])
print(raw_order_products[raw_order_products['product_weight_g']<0])
print(raw_order_products[raw_order_products['product_length_cm']<0])
print(raw_order_products[raw_order_products['product_height_cm']<0])
print(raw_order_products[raw_order_products['product_width_cm']<0])

Empty DataFrame
Columns: [product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm]
Index: []
Empty DataFrame
Columns: [product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm]
Index: []
Empty DataFrame
Columns: [product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm]
Index: []
Empty DataFrame
Columns: [product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight_g, product_length_cm, product_height_cm, product_width_cm]
Index: []
Empty DataFrame
Columns: [product_id, product_category_name, product_name_lenght, product_description_lenght, product_photos_qty, product_weight

In [42]:
# Q: Any category names containing digits (possible data error)?
raw_order_products[raw_order_products['product_category_name'].astype(str).str.contains(r'\d', na=False)]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
339,e5d860a0fff2e94bd1a406456cc6c4b3,eletrodomesticos_2,38.0,2030.0,2.0,13200.0,66.0,19.0,67.0
565,28a4dc65b49348e9de7f1aa7bfbabfd8,eletrodomesticos_2,59.0,319.0,7.0,14350.0,41.0,55.0,31.0
904,cc3181a81384b6fc34602d49cb6068a6,eletrodomesticos_2,33.0,94.0,1.0,18450.0,68.0,21.0,42.0
1055,418b40c9e83b7e5ac79e4a7ef975ebbe,eletrodomesticos_2,47.0,796.0,3.0,7300.0,82.0,10.0,54.0
2393,782447ddd6d2ba5b64cb23ef7a8db626,eletrodomesticos_2,44.0,57.0,1.0,3585.0,33.0,6.0,19.0
...,...,...,...,...,...,...,...,...,...
31471,28dc766929435d5a8555ce85e2f82667,eletrodomesticos_2,32.0,679.0,1.0,12783.0,36.0,52.0,31.0
31646,fdcd405125af2d9b2b7b6fa42950fd92,eletrodomesticos_2,34.0,264.0,1.0,450.0,60.0,20.0,20.0
31983,46f8312de289b5679ef6c699c028b572,eletrodomesticos_2,55.0,1302.0,4.0,500.0,20.0,16.0,20.0
32403,b1370e983f7a817244be390d1f7f2e7b,eletrodomesticos_2,51.0,792.0,1.0,22600.0,51.0,41.0,62.0


## Category Translation

In [43]:
raw_category_translation=pd.read_csv('../data/raw/product_category_name_translation.csv')
print(raw_category_translation.head(5))

    product_category_name product_category_name_english
0            beleza_saude                 health_beauty
1  informatica_acessorios         computers_accessories
2              automotivo                          auto
3         cama_mesa_banho                bed_bath_table
4        moveis_decoracao               furniture_decor


In [44]:
# Q: Any category names (source or English) containing digits?
print(raw_category_translation['product_category_name'].astype(str).str.contains(r'\d', na=False))
print(raw_category_translation['product_category_name_english'].astype(str).str.contains(r'\d', na=False))

0     False
1     False
2     False
3     False
4     False
      ...  
66    False
67    False
68    False
69    False
70    False
Name: product_category_name, Length: 71, dtype: bool
0     False
1     False
2     False
3     False
4     False
      ...  
66    False
67    False
68    False
69    False
70    False
Name: product_category_name_english, Length: 71, dtype: bool


In [45]:
# Q: Is the mapping clean -- one English name per source category, no nulls/dupes?
print(raw_category_translation['product_category_name'].nunique())
print(raw_category_translation['product_category_name_english'].nunique())
print(raw_category_translation.shape)
print(raw_category_translation.isna().sum())
print(raw_category_translation.duplicated().sum())
# Finding: 71 rows, no nulls, no duplicates, 71 unique category names mapping to 71
# unique English names.

71
71
(71, 2)
product_category_name            0
product_category_name_english    0
dtype: int64
0


In [46]:
# Q: Are there any product categories with no English translation available at all?
print(set(raw_order_products['product_category_name'])-set(raw_category_translation['product_category_name']))
# Finding: 2 categories present in products are missing from this table entirely:
# pc_gamer and portateis_cozinha_e_preparadores_de_alimentos -- need manual
# translation.

{'pc_gamer', nan, 'portateis_cozinha_e_preparadores_de_alimentos'}


In [47]:
print(raw_category_translation[raw_category_translation['product_category_name'].astype(str).str.contains(r'\d', na=False)])
print(raw_category_translation[raw_category_translation['product_category_name_english'].astype(str).str.contains(r'\d', na=False)])

   product_category_name product_category_name_english
56    eletrodomesticos_2             home_appliances_2
62       casa_conforto_2                home_comfort_2
   product_category_name product_category_name_english
56    eletrodomesticos_2             home_appliances_2
62       casa_conforto_2                home_comfort_2


In [48]:
# Q: How many products are actually affected by the 2 missing translations?
print(raw_order_products[raw_order_products['product_category_name'].isin(['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'])].shape)
# Finding: 13 products affected.

(13, 9)


## Sellers

In [49]:
raw_sellers=pd.read_csv('../data/raw/olist_sellers_dataset.csv')

# Q: What does the sellers table look like structurally?
print(raw_sellers.head(5))
print(raw_sellers.shape)
print(raw_sellers.info())
print(raw_sellers.isna().sum())
# Finding: 3,095 rows, 4 columns, zero nulls.

                          seller_id  seller_zip_code_prefix  \
0  3442f8959a84dea7ee197c632cb2df15                   13023   
1  d1b65fc7debc3361ea86b5f14c68d2e2                   13844   
2  ce3ad9de960102d0677a81f5d0bb7b2d                   20031   
3  c0f3eea2e14555b6faeea3dd58c1b1c3                    4195   
4  51a04a8a6bdcb23deccc82b0b80742cf                   12914   

         seller_city seller_state  
0           campinas           SP  
1         mogi guacu           SP  
2     rio de janeiro           RJ  
3          sao paulo           SP  
4  braganca paulista           SP  
(3095, 4)
<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   

In [50]:
# Q: Any blank-string city/state values?
print(raw_sellers[raw_sellers['seller_city'].dropna().str.strip()==''])
print(raw_sellers[raw_sellers['seller_state'].dropna().str.strip()==''])
# Finding: zero blank strings.

Empty DataFrame
Columns: [seller_id, seller_zip_code_prefix, seller_city, seller_state]
Index: []
Empty DataFrame
Columns: [seller_id, seller_zip_code_prefix, seller_city, seller_state]
Index: []


In [51]:
print(raw_sellers['seller_state'].nunique())
print(raw_sellers['seller_city'].nunique())

23
611


In [52]:
# Q: Do sellers exist in every state that customers are in?
print(set(raw_customers['customer_state'])-set(raw_sellers['seller_state']))
# Finding: 23 unique seller states vs. customers' 27 -- sellers are absent from TO,
# AL, RR, AP. A real business fact, not a data error.

{'RR', 'AP', 'AL', 'TO'}


In [53]:
# Q: Is seller_id unique, and are there duplicate rows?
print(raw_sellers['seller_id'].value_counts().sort_values(ascending=False))
print(raw_sellers.duplicated().sum())
# Finding: zero duplicates, zero blank strings.

seller_id
3442f8959a84dea7ee197c632cb2df15    1
d1b65fc7debc3361ea86b5f14c68d2e2    1
ce3ad9de960102d0677a81f5d0bb7b2d    1
c0f3eea2e14555b6faeea3dd58c1b1c3    1
51a04a8a6bdcb23deccc82b0b80742cf    1
                                   ..
98dddbc4601dd4443ca174359b237166    1
f8201cab383e484733266d1906e2fdfa    1
74871d19219c7d518d0090283e03c137    1
e603cf3fec55f8697c9059638d6c8eb5    1
9e25199f6ef7e7c347120ff175652c3b    1
Name: count, Length: 3095, dtype: int64
0


In [54]:
# Q: Is the zip prefix range sane?
print(raw_sellers['seller_zip_code_prefix'].min())
print(raw_sellers['seller_zip_code_prefix'].max())
# Finding: zip prefix range 1,001-99,730 -- same leading-zero truncation issue as
# customers.

1001
99730


In [55]:
# Q: Any city/state values containing digits?
print(raw_sellers['seller_city'][raw_sellers['seller_city'].str.contains(r'\d', regex=True)])
print(raw_sellers['seller_state'][raw_sellers['seller_state'].str.contains(r'\d', regex=True)])
# Finding: one row has seller_city = '04482255' -- a zip code sitting in the city
# field, not a real city name. Flagged for cleaning.

517    04482255
Name: seller_city, dtype: str
Series([], Name: seller_state, dtype: str)


## Geolocation

In [56]:
raw_geolocation=pd.read_csv('../data/raw/olist_geolocation_dataset.csv')

# Q: What does the geolocation table look like structurally?
print(raw_geolocation.head(5))
print(raw_geolocation.shape)
print(raw_geolocation.info())
print(raw_geolocation.isna().sum())
# Finding: 1,000,163 rows, 5 columns, zero nulls.

   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                         1037       -23.545621       -46.639292   
1                         1046       -23.546081       -46.644820   
2                         1046       -23.546129       -46.642951   
3                         1041       -23.544392       -46.639499   
4                         1035       -23.541578       -46.641607   

  geolocation_city geolocation_state  
0        sao paulo                SP  
1        sao paulo                SP  
2        sao paulo                SP  
3        sao paulo                SP  
4        sao paulo                SP  
(1000163, 5)
<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2 

In [57]:
# Q: How many unique zip prefixes, and how many raw rows per prefix?
print(raw_geolocation['geolocation_zip_code_prefix'].nunique())
print(raw_geolocation.groupby('geolocation_zip_code_prefix').size().describe())
# Finding: 19,015 unique zip prefixes; row count per prefix ranges 1-1,146.

19015
count    19015.000000
mean        52.598633
std         72.057907
min          1.000000
25%         10.000000
50%         29.000000
75%         66.500000
max       1146.000000
dtype: float64


In [58]:
# Q: Are there full-row duplicates?
print(raw_geolocation.duplicated().sum())
# Finding: 261,831 full-row duplicates -- genuinely redundant, safe to drop.

261831


In [59]:
# Q: Are all lat/lng values within Brazil's real bounding box?
print(raw_geolocation['geolocation_lat'].min(), raw_geolocation['geolocation_lat'].max())
print(raw_geolocation['geolocation_lng'].min(), raw_geolocation['geolocation_lng'].max())
# Finding: values far outside Brazil's actual bounding box exist (e.g. lat 45.07,
# lng 121.11) -- 8 rows affected. A generic +-90/+-180 range check would miss these
# entirely, so a Brazil-specific bounding box is needed.

-36.6053744107061 45.06593318269697
-101.46676644931476 121.10539381057764


In [60]:
# Q: Are city/state fields consistent per zip prefix?
print(raw_geolocation['geolocation_state'].nunique())  # expect 27
print(raw_geolocation['geolocation_state'].value_counts())
print(raw_geolocation['geolocation_city'].str.strip().str.lower().nunique())
print(raw_geolocation['geolocation_city'].nunique())
# Finding: 8,556 of 19,015 prefixes have more than one distinct geolocation_city
# value -- investigated and confirmed to be pure accent/encoding noise
# (sao paulo vs sao paulo), not real geographic conflicts. Whether any prefixes have
# more than one distinct geolocation_state is NOT yet checked at this audit stage --
# this matters more than the city conflict, since any lat/lng aggregation by prefix
# needs to avoid averaging across genuinely different states.

27
geolocation_state
SP    404268
MG    126336
RJ    121169
RS     61851
PR     57859
SC     38328
BA     36045
GO     20139
ES     16748
PE     16432
DF     12986
MT     12031
CE     11674
PA     10853
MS     10431
MA      7853
PB      5538
RN      5041
PI      4549
AL      4183
TO      3576
SE      3563
RO      3478
AM      2432
AC      1301
AP       853
RR       646
Name: count, dtype: int64
8010
8011


In [61]:
# Q: Do customer/seller zip prefixes all have a matching geolocation row?
print(set(raw_customers['customer_zip_code_prefix'].unique()) - set(raw_geolocation['geolocation_zip_code_prefix'].unique()))
print(set(raw_sellers['seller_zip_code_prefix'].unique()) - set(raw_geolocation['geolocation_zip_code_prefix'].unique()))
# Finding: 8 customer zip prefixes and 7 seller zip prefixes have no matching row in
# geolocation at all.

{np.int64(28160), np.int64(56327), np.int64(75784), np.int64(29196), np.int64(71698), np.int64(68629), np.int64(29718), np.int64(70686), np.int64(67105), np.int64(73255), np.int64(71208), np.int64(12332), np.int64(70701), np.int64(70702), np.int64(72238), np.int64(72237), np.int64(7729), np.int64(72242), np.int64(72243), np.int64(64047), np.int64(55863), np.int64(72760), np.int64(73272), np.int64(70716), np.int64(72268), np.int64(35408), np.int64(72280), np.int64(36956), np.int64(2140), np.int64(71261), np.int64(73310), np.int64(77404), np.int64(76897), np.int64(64095), np.int64(64605), np.int64(44135), np.int64(7784), np.int64(72300), np.int64(95853), np.int64(65137), np.int64(72821), np.int64(86135), np.int64(85118), np.int64(71810), np.int64(41098), np.int64(37005), np.int64(84623), np.int64(72338), np.int64(72341), np.int64(8342), np.int64(73369), np.int64(59547), np.int64(72863), np.int64(62625), np.int64(94370), np.int64(72867), np.int64(56485), np.int64(76968), np.int64(70316), 

In [62]:
# Q: What would a simple one-row-per-prefix lookup table look like? (early look --
# finalized properly with the bounding-box and state-conflict filters in
# 02_data_cleaning)
geo_lookup = raw_geolocation.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'mean'),
    lng=('geolocation_lng', 'mean'),
    city=('geolocation_city', lambda x: x.mode()[0] if not x.mode().empty else np.nan),
    state=('geolocation_state', lambda x: x.mode()[0] if not x.mode().empty else np.nan)
).reset_index()
print(geo_lookup.shape)

(19015, 5)


In [63]:
# Q: Do any zip prefixes span more than one distinct state? (the check flagged as
# missing two cells up)
raw_geolocation.groupby('geolocation_zip_code_prefix')['geolocation_state'].nunique().sort_values(ascending=False).head(10)

state_conflicts = raw_geolocation.groupby('geolocation_zip_code_prefix')['geolocation_state'].nunique()
print((state_conflicts > 1).sum())
# NOTE: this cell originally had a stray incomplete line ('state') left over from
# exploration -- removed here since it would raise a NameError. The state-conflict
# check above is the real point, and is resolved properly (filtered to each prefix's
# modal state before averaging lat/lng) in 02_data_cleaning.

8


## Cross-Table Checks

In [64]:
# Q: Does every customer have an order, and every order a customer?
custNoOrder=~raw_customers['customer_id'].isin(raw_orders['customer_id'])
print(custNoOrder.sum())
print(raw_customers[custNoOrder])

orderNoCustomer=~raw_orders['customer_id'].isin(raw_customers['customer_id'])
print(orderNoCustomer.sum())
print(raw_orders[orderNoCustomer])
# Finding: fully consistent both directions (0 mismatches).

0
Empty DataFrame
Columns: [customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state]
Index: []
0
Empty DataFrame
Columns: [order_id, customer_id, order_status, order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date]
Index: []


In [65]:
# Q: Does every order_item have a matching order, and does every order have at
# least one item?
ItemWithNoOrder=~raw_order_items['order_id'].isin(raw_orders['order_id'])
print(ItemWithNoOrder.sum())

orderWithNoItem=~raw_orders['order_id'].isin(raw_order_items['order_id'])
print(orderWithNoItem.sum())
print(raw_orders[orderWithNoItem].groupby('order_status')['order_status'].value_counts())
# only canceled, unavailable, created, invoiced -- just 1 is 'shipped'

print(raw_orders[orderWithNoItem & (raw_orders['order_status']=='shipped')])
# Finding: 775 orders have no items -- all explained by non-delivered statuses,
# except 1 'shipped' order with no items, no carrier date, no customer date --
# a genuine incomplete record.

0
775
order_status
canceled       164
created          5
invoiced         2
shipped          1
unavailable    603
Name: count, dtype: int64
                               order_id                       customer_id  \
23254  a68ce1686d536ca72bd2dadc4b8671e5  d7bed5fac093a4136216072abaf599d5   

      order_status order_purchase_timestamp   order_approved_at  \
23254      shipped      2016-10-05 01:47:40 2016-10-07 03:11:22   

      order_delivered_carrier_date order_delivered_customer_date  \
23254          2016-11-07 16:37:37                           NaT   

      order_estimated_delivery_date  
23254                    2016-12-01  


In [66]:
# Q: Are order_items fully consistent with products and sellers?
itemWithNoPdt=~raw_order_items['product_id'].isin(raw_order_products['product_id'])
print(itemWithNoPdt.sum())
pdtWithNoItem=~raw_order_products['product_id'].isin(raw_order_items['product_id'])
print(pdtWithNoItem.sum())

itemWithNoSeller=~raw_order_items['seller_id'].isin(raw_sellers['seller_id'])
print(itemWithNoSeller.sum())
sellerWithNoItem=~raw_sellers['seller_id'].isin(raw_order_items['seller_id'])
print(sellerWithNoItem.sum())
# Finding: fully consistent (0 mismatches) in both directions, for both the
# product and seller relationships.

0
0
0
0


In [67]:
# Q: Does every payment have a matching order, and does every order have a payment?
paymentMissingOrder = ~raw_order_payments['order_id'].isin(raw_orders['order_id'])
print(paymentMissingOrder.sum())
orderMissingPayment = ~raw_orders['order_id'].isin(raw_order_payments['order_id'])
print(orderMissingPayment.sum())

raw_orders[~raw_orders['order_id'].isin(raw_order_payments['order_id'])]
# Finding: 0 payments without a matching order; 1 order has no matching payment
# record -- needs inspection before deciding how to handle.

0
1


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
30710,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04


In [68]:
# Q: Does every review have a matching order?
missing = ~raw_order_reviews['order_id'].isin(raw_orders['order_id'])
print(missing.sum())
# Finding: fully consistent (0 mismatches).

0


## Open items carried into cleaning

1. Zip code leading-zero truncation (customers, sellers, geolocation).
2. Seller row with a zip code in the city field.
3. Geolocation state-conflict check (not yet run here).
4. `freight_value > price` pattern not yet verified against category/price.
5. Payments-vs-order_items cross-table reconciliation not yet run.
6. The 1 order missing a payment record -- needs inspection.
7. The 171-day order-approval outlier -- needs a manual look.